# Thermochemistry of all saved apatinib routes

This tutorial reuses **all 435 pre-generated apatinib routes and their 1,192 steps**, in saved order. It preserves the original route IDs, search scores, and stored forward annotations. Planning and forward-model inference are not rerun. The independent thermochemistry report displays search scores, five scientific values per step, explicit filter/error states, and one click-open Manifest; it does not display forward-validation results or policy log likelihood.

`data/apatinib_routes.provenance.json` records the source collection and SHA-256. Set `SYNPLAN_AIMNET_ROUTES` to another saved route file if needed. Exhaustive stereo enumeration has no cap; the full CPU run can take substantial time. Completed reactions and successful molecules are checkpointed.

## Installation

Use Python 3.11+. From the repository root:

```bash
python -m pip install 'torch>=2.10' --index-url https://download.pytorch.org/whl/cpu
python -m pip install -e '.[cpu]'
python -m pip install 'loqi @ git+https://github.com/isayevlab/LoQI@04f3caef4521b3fbcb236f8b01f4bc80c83cb918'
```

For an existing uv environment, use `uv sync --extra cpu --inexact` to retain both CPU backend selection and the separately installed LoQI package. The setup cell discovers the checkout from the repository root, `tutorials`, or `docs/user_guide`.

Model registries verify/download weights on first use. Set `SYNPLAN_AIMNET_MODEL` and `SYNPLAN_LOQI_MODEL` to local checkpoint files for offline execution. Installation is intentionally documentation rather than executable notebook cells.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import torch

repository = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "pyproject.toml").is_file()
     and (candidate / "synplan").is_dir()
     and (candidate / "tutorials/data/apatinib_routes.json").is_file()),
    None,
)
if repository is None:
    raise RuntimeError("Open this notebook from the SynPlanner checkout or one of its subdirectories.")
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))
from scripts.assess_thermochemistry import load_routes, run_thermochemistry_assessment
from synplan.chem.reaction.routes.io import read_routes_json
from synplan.chem.reaction.score import AIMNetConfig
from synplan.chem.reaction.routes.thermochemistry_report import display_status

threads = 4
torch.set_num_threads(threads)
routes_path = Path(os.environ.get("SYNPLAN_AIMNET_ROUTES", str(repository / "tutorials/data/apatinib_routes.json")))
results_dir = Path(os.environ.get("SYNPLAN_AIMNET_RESULTS", "tutorial_results/aimnet_apatinib"))
cache_path = Path(os.environ.get("SYNPLAN_AIMNET_CACHE", str(results_dir / "aimnet_cache.json")))
labels, routes = load_routes(routes_path)
before = [r.to_json() for r in routes]
print(f"{len(routes)} routes; {sum(len(r) for r in routes)} step occurrences")


## Assess and checkpoint

The next cell loads `configs/aimnet_thermo.yaml` through `AIMNetConfig.from_yaml()`. Edit that file for custom scientific settings; explicit local-checkpoint environment variables override its checkpoint paths. Relative paths in YAML resolve from the working directory.

Defaults remain CPU, ten LoQI conformers per stereoisomer, 25 flow steps, BFGS at 0.01 eV/Å with 200 steps maximum, 298.15 K, and 101325 Pa. Rotational symmetry defaults to 1. An isolated proton uses electronic energy 0 eV plus monatomic thermal contributions; this convention appears in the Manifest.

Set `SYNPLAN_AIMNET_WORKERS` to use several isolated CPU workers. Each worker uses four threads and species-specific seeds; the parent alone writes the shared molecular cache. The default is one worker. An identical interrupted run resumes from completed reaction checkpoints. Use a new results directory to retry errored reactions, retaining the same molecular cache. Reaction/worker checkpoint format 2 cannot resume older reaction checkpoints: set `SYNPLAN_AIMNET_RESULTS` to a new directory and `SYNPLAN_AIMNET_CACHE` to a copy of the prior molecular cache. A failed resume startup leaves the old report intact and writes a separate attempt log.

These are gas-phase reference equations, without solvent or ensemble corrections. For the chloride substitution, the balanced products include separate `[H+]` and `[Cl-]`; the energies describe that ionic equation.


In [ ]:
def local_checkpoint(name):
    value = os.environ.get(name)
    return Path(value).resolve() if value else None

config = AIMNetConfig.from_yaml(str(repository / "configs/aimnet_thermo.yaml"))
settings = config.model_dump()
for field, variable in (("aimnet_checkpoint", "SYNPLAN_AIMNET_MODEL"),
                        ("loqi_checkpoint", "SYNPLAN_LOQI_MODEL")):
    if path := local_checkpoint(variable):
        settings[field] = path
config = AIMNetConfig.model_validate(settings)
run_manifest = run_thermochemistry_assessment(
    routes_path, results_dir, config, cache_path,
    workers=int(os.environ.get("SYNPLAN_AIMNET_WORKERS", "1")),
    threads=threads, verify_cache=True,
)
assessed_routes = read_routes_json(results_dir / "routes_aimnet.json", as_routes=True)
assert [r.to_json() for r in routes] == before
assert [r.provenance for r in assessed_routes] == [r.provenance for r in routes]
assert [s.forward_prediction for r in assessed_routes for s in r] == [s.forward_prediction for r in routes for s in r]
run_manifest["statistics"]


## Inspect step coverage

Every step remains in its route. **Processed** means all variants have thermochemistry. **Filtered out** identifies an explicit applicability/confidence policy. **Errored** identifies a preparation or calculation failure; partially calculated steps retain successful values and explicit coverage. A short Log under an errored row can be copied or downloaded.

Each rendered route is followed by a table of ΔE, ΔH, ΔG, balancing confidence, and calculated/total variants. Multiple variants use labeled min-max ranges over successful results, with all individual variants expandable. Unavailable values are shown as “—”.


In [ ]:
rows = []
for route_id, route in zip(labels, assessed_routes):
    for step_id, step in enumerate(route, 1):
        annotation = step.thermochemistry
        rows.append({
            "route": route_id, "step": step_id,
            "status": display_status(annotation),
            "calculated variants": annotation.get("successful_variants", 0),
            "total variants": len(annotation["variants"]),
        })
coverage = pd.DataFrame(rows)
assert len(coverage) == sum(len(r) for r in routes)
coverage.groupby("status").size().rename("steps")


## Report, complete data, and cached rerun

`routes_aimnet.html` is the independent report. Configuration, calculator/model identities, source provenance, timing, and cache statistics appear once in its top-right Manifest. Temperature, pressure, and symmetry assumptions remain visible. `routes_aimnet.json` retains complete annotations, including molecular coordinates and frequencies, for machine use.

The runner reopens the molecular cache and re-assesses all previously successful reactions with new model calculations forbidden. Expected molecular failures remain explicit and are not treated as cached energies.


In [ ]:
warm = run_manifest["cached_rerun"]
assert warm["molecular_calculations"] == 0
print("Cached verification:", warm["reaction_occurrences"], "successful reaction occurrences;", warm["cache_hits"], "cache hits")
print((results_dir / "routes_aimnet.html").resolve())
print((results_dir / "routes_aimnet.json").resolve())
